# 04 — Troubleshooting with the Spark UI

Three classic production problems, reproduced on purpose so you can see them in the UI:

| # | Problem | Symptom in the UI | Fix to try |
|---|---|---|---|
| 1 | **Memory pressure** | Stage shows *Spill (memory)* / *Spill (disk)*, long GC time | More partitions, fewer bytes per row |
| 2 | **Data skew** | One task in a stage runs far longer than the rest (Max ≫ Median) | AQE skew join, salting |
| 3 | **Slow joins** | Huge *shuffle write* on both sides of a join | Broadcast the small side |

### Where to look

- **Spark UI (live):** use the **Spark UI** link under the cell output, or the **Spark UI** tab on the attached cluster's page.
- **Spark UI (finished runs):** each run's Spark UI stays reachable from the cluster page and the job run page. Compare runs there after the fact.
- **Tables** are Delta tables in the catalog and schema you set in the widgets at the top of the notebook.

**Attach this notebook to a classic (all-purpose) cluster, not serverless.** The lab sets AQE and broadcast options and sizes memory on purpose; serverless doesn't allow those settings, so the experiments wouldn't reproduce. Use a Databricks Runtime LTS with Spark 3.5 or later.

For each scenario, open the **Jobs**, **Stages**, **Executors** and **SQL / DataFrame** tabs and compare the run with the "after" run.

## 0 — Session and helpers

In [ ]:
import time
from pyspark.sql import functions as F

# `spark` is already provided by the cluster. Set these in the widgets above the notebook.
print(f'Spark {spark.version}')
print('Spark UI: open it from the cluster menu (Spark UI) or the Spark UI link in the cell output.')

In [ ]:
dbutils.widgets.text('catalog', 'main', 'Catalog')
dbutils.widgets.text('schema', 'spark_ui_lab', 'Schema')
NS = f"{dbutils.widgets.get('catalog')}.{dbutils.widgets.get('schema')}"
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {NS}')

def timed(label, df):
    """Run the query to completion (noop sink, no rows returned to the driver) and print wall time."""
    t0 = time.perf_counter()
    df.write.format('noop').mode('overwrite').save()
    print(f'{label}: {time.perf_counter() - t0:.1f}s')

def reset_conf():
    """Put the knobs back to the defaults from spark-defaults.conf between scenarios."""
    spark.conf.unset('spark.sql.autoBroadcastJoinThreshold')
    spark.conf.unset('spark.sql.shuffle.partitions')
    spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', 'true')
    spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')

## 1 — Build the test data (Delta)

- `customers`: 200k rows, one per `customer_id`.
- `orders_uniform`: 4M orders spread evenly over all customers.
- `orders_skewed`: same orders, but **70%** of them point at customer `0` (a "hot" key — think one big merchant or a bot account).

Each order carries 128 random bytes of payload. Random bytes don't compress, so the data is as large on disk and in shuffle as it is in memory. Repeated text would compress to almost nothing and the symptoms below would disappear.

Change the knobs here to make the problems bigger or smaller. The defaults are tuned for a small classic cluster (for example two workers with 4 cores and 16 GB each). If a scenario shows no spill, raise the row count; if it spills too much, lower it.

In [ ]:
N_ORDERS = 4_000_000
N_CUSTOMERS = 200_000
HOT_SHARE = 0.7

if not spark.catalog.tableExists(f'{NS}.customers'):
    customers = (
        spark.range(N_CUSTOMERS)
        .select(
            F.col('id').alias('customer_id'),
            F.concat(F.lit('name_'), F.col('id').cast('string')).alias('name'),
            F.expr("element_at(array('US','UK','MX','SG','DE'), cast(id % 5 as int) + 1)").alias('country'),
        )
    )
    customers.writeTo(f'{NS}.customers').using('delta').createOrReplace()

    base = spark.range(N_ORDERS).select(
        F.col('id').alias('order_id'),
        (F.rand(seed=1) * N_CUSTOMERS).cast('long').alias('customer_id'),
        (F.rand(seed=2) * 500).alias('amount'),
        # 4 x 32 random bytes = 128 bytes per row, incompressible
        F.concat(*[F.unhex(F.sha2(F.concat(F.col('id').cast('string'), F.lit(f'-{i}')), 256)) for i in range(4)]).alias('payload'),
    )
    # 32 input files so the scan runs as 32 tasks, not 2
    base.repartition(32).writeTo(f'{NS}.orders_uniform').using('delta').createOrReplace()

    (base
        .withColumn('customer_id', F.when(F.rand(seed=3) < HOT_SHARE, F.lit(0)).otherwise(F.col('customer_id')))
        .repartition(32)
        .writeTo(f'{NS}.orders_skewed').using('delta').createOrReplace())

for t in ['customers', 'orders_uniform', 'orders_skewed']:
    print(t, spark.table(f'{NS}.{t}').count())

In [ ]:
# Delta keeps file-level details — handy to check layout before you test.
spark.sql(f'DESCRIBE DETAIL {NS}.orders_skewed').select('numFiles', 'sizeInBytes').show()

## 2 — Memory pressure

A sort (or an aggregation with large per-key state) has to hold its input in **execution memory**. When a task's share of that memory runs out, Spark spills sorted runs to local disk and then merges them back, which is slow and burns GC time.

Execution memory per task is roughly the executor's unified memory divided by its task slots. On the classic cluster in this notebook, one task gets about 1 GB at most. The two runs below sort the same 2M wide rows (~4 GB), with a filter rather than a `limit()`: a `limit()` before a sort gathers everything into one partition, which hides the partition count you are testing. The only difference is the number of shuffle partitions. AQE coalescing is off in both, so the partition count you ask for is the one you get:

- **2A — under-partitioned:** 4 partitions, so each task sorts ~1 GB and must spill.
- **2B — right-sized:** 200 partitions, so each task sorts ~20 MB and stays in memory.

**In the UI:** Stages → click the sort stage → the *Spill (Memory)* and *Spill (Disk)* columns, and the *GC Time* in the task table. Executors tab → *Peak JVM Memory* and *GC Time*.

In [ ]:
reset_conf()
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', 'false')  # keep the partition count we asked for

wide = (
    spark.table(f'{NS}.orders_uniform')
    .where(F.col('order_id') < 2_000_000)
    .withColumn('payload_wide', F.concat(*[F.col('payload')] * 16))  # 2 KB per row
    .orderBy(F.rand(seed=7))
)

spark.conf.set('spark.sql.shuffle.partitions', '4')
timed('2A under-partitioned (4 partitions)', wide)

spark.conf.set('spark.sql.shuffle.partitions', '200')
timed('2B right-sized (200 partitions)', wide)

**Takeaways:** more partitions cut the per-task memory need. Other levers: select only the columns you need *before* a wide shuffle (`payload_wide` above is 16× the size of the rest of the row), or give executors more memory.

## 3 — Data skew

`orders_skewed` has one customer with 70% of the orders. In a shuffle join or `groupBy` on `customer_id`, every row with the same key goes to the **same** task, so one task gets ~70% of the rows while the others get a handful each.

**In the UI:** Stages → the join/aggregate stage → the *Summary Metrics* table: compare **Min / Median / Max** of *Duration* and *Shuffle Read Size*. A Max that's 50–100× the Median is skew. The SQL tab shows the same join with the AQE skew-join node once it's enabled.

In [ ]:
reset_conf()
# Force a shuffle join: take the broadcast option off the table so the skew is visible.
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')

skewed = spark.table(f'{NS}.orders_skewed')
customers = spark.table(f'{NS}.customers')

# Simple skew check: rows per key (should show customer 0 at the top)
skewed.groupBy('customer_id').count().orderBy(F.desc('count')).show(5)

# Keep every column in the join output (no aggregation), so the payload is shuffled as well.
# An aggregation on top would prune the payload and the skew would never show up.
joined = skewed.join(customers, 'customer_id')

spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'false')
timed('3A skewed join, AQE skew handling OFF', joined)

spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')
timed('3B skewed join, AQE skew handling ON', joined)

**Takeaways:** with AQE skew handling on, Spark splits the oversized partition into several smaller tasks (look for `skew=true` in the SQL plan under the `AQEShuffleRead` node). If AQE can't help (for example the skew is in a `groupBy`), the fallback is **salting**: append a random suffix to the hot key, aggregate in two steps, then drop the salt.

### Salting: fixing skew by hand

When AQE can't split the hot partition (for example, the skew is in a `groupBy`, or the join is not an inner equi-join), you can **salt** the key yourself:

1. Give each row of the skewed table a random `salt` between 0 and 15, so the hot key `0` is spread over 16 keys: `(0, 0)` … `(0, 15)`.
2. Copy the small table once per salt value, so every salted key still finds its match.
3. Join on `(customer_id, salt)`. The hot key now becomes 16 smaller partitions that run in parallel.

The cost is the copy in step 2: the customers table is 16× larger. Salt only the hot keys in production, and handle the rest of the keys without salt.

**In the UI:** compare this join with 3A. The join stage should show no spill, and the max task time should be close to the median.

In [ ]:
SALT_BUCKETS = 16

reset_conf()
# Turn AQE skew handling OFF, so this run shows what salting does on its own.
spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'false')
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')

salted_orders = skewed.withColumn('salt', (F.rand(seed=11) * SALT_BUCKETS).cast('int'))
salts = spark.range(SALT_BUCKETS).select(F.col('id').cast('int').alias('salt'))
salted_customers = customers.crossJoin(salts)  # one copy of each customer per salt value

salted_join = salted_orders.join(salted_customers, ['customer_id', 'salt'])

# Same row count as the unsalted join: salting must not change the result.
print('rows, unsalted:', joined.count(), '| rows, salted:', salted_join.count())
timed('3C salted join (16 buckets), AQE skew handling OFF', salted_join)

reset_conf()

## 4 — Slow joins

Same join, same data, different strategy:

- **4A — sort-merge join:** both sides are shuffled on `customer_id` and sorted. Expensive when both sides are big.
- **4B — broadcast hash join:** the 200k-row `customers` table is copied to every executor, and `orders` is not shuffled at all.

**In the UI:** SQL / DataFrame tab → open each query. Look at the physical plan: *SortMergeJoin + Exchange* on both sides vs *BroadcastHashJoin* with a *BroadcastExchange* on the small side. Compare *shuffle bytes written* and the total duration.

In [ ]:
reset_conf()
orders = spark.table(f'{NS}.orders_uniform')
customers = spark.table(f'{NS}.customers')

spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
# Same reason as section 3: no aggregation on top, so both joins move the full rows.
sort_merge = orders.join(customers, 'customer_id')
timed('4A sort-merge join', sort_merge)

broadcast = orders.join(F.broadcast(customers), 'customer_id')
timed('4B broadcast join', broadcast)

In [ ]:
# Read the plans: look for SortMergeJoin/Exchange vs BroadcastHashJoin/BroadcastExchange.
print('--- 4A plan ---')
orders.join(customers, 'customer_id').explain('formatted')
print('--- 4B plan ---')
orders.join(F.broadcast(customers), 'customer_id').explain('formatted')

**Takeaways:** broadcast the side that fits comfortably in executor memory (the default auto-broadcast threshold is 10 MB; check it with `spark.conf.get('spark.sql.autoBroadcastJoinThreshold')`). If a join is slow and both inputs are big, look first at whether a filter can be pushed below the join, and at whether the join key is skewed (section 3).

Compare the run labels in the **Spark UI** of this cluster (SQL / DataFrame tab) to see all of the above side by side.

In [ ]:
reset_conf()
print('Reset to the spark-defaults.conf values.')